# 6. Klastering Tipologi Pasar
Mengelompokkan pasar dengan K-Means berdasarkan harga relatif terhadap median nasional, inflasi, volatilitas inflasi, dan rentang harga selama 12 bulan terakhir.

In [ ]:
import re

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, f1_score, silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeClassifier

import os
import matplotlib.pyplot as plt

data_path = '../data/processed/data_bersih.csv'
output_dir = '../data/processed'
KOL_PROVINSI = 'adm1_name'

In [ ]:
KOL_PASAR = 'mkt_name'
KOL_TANGGAL = 'date'
KOL_INDEKS = 'food_price_index'
KOMODITAS = ['eggs', 'garlic', 'meat_beef', 'meat_chicken', 'meat_chicken_broiler', 'oil', 'onions', 'rice', 'sugar']

BATAS_WASPADA = 5
BATAS_LONJAKAN = 10
NAMA_KELAS = ['Normal', 'Waspada', 'Lonjakan']
FITUR_DASAR = ['inflasi_ini', 'inflasi_lalu', 'bulan']
PERSEN_TEST = 0.2
TAHUN_ACUAN_AKHIR = 2019

BOBOT_RISIKO = 0.6
BOBOT_MOMENTUM = 0.2
BOBOT_VOLATILITAS = 0.2
BATAS_PRIORITAS = 50
BATAS_PANTAU = 25
BATAS_HARGA_PERSEN = 5

NAMA_KOMODITAS = {
    'eggs': 'Telur', 'garlic': 'Bawang Putih', 'meat_beef': 'Daging Sapi', 'meat_chicken': 'Daging Ayam',
    'meat_chicken_broiler': 'Daging Ayam Broiler', 'oil': 'Minyak Goreng', 'onions': 'Bawang Merah',
    'rice': 'Beras', 'sugar': 'Gula', 'food_price_index': 'Indeks Pangan',
}
MUSIM_ONI = ['DJF', 'JFM', 'FMA', 'MAM', 'AMJ', 'MJJ', 'JJA', 'JAS', 'ASO', 'SON', 'OND', 'NDJ']

In [ ]:
def tambah_rentang(d):
    h, l, c = f'h_{KOL_INDEKS}', f'l_{KOL_INDEKS}', f'c_{KOL_INDEKS}'
    if {h, l, c}.issubset(d.columns):
        return (d[h] - d[l]) / d[c] * 100
    return pd.Series(np.nan, index=d.index)


def klaster_pasar(df, bulan_terakhir=12, k_min=3, k_max=6):
    harga = [f'c_{k}' for k in KOMODITAS if f'c_{k}' in df.columns]
    kol_inf = f'inflation_{KOL_INDEKS}'
    akhir = df[KOL_TANGGAL].max()
    r = df[df[KOL_TANGGAL] > akhir - pd.DateOffset(months=bulan_terakhir)].copy()
    median_bulan = r.groupby(KOL_TANGGAL)[harga].transform('median')
    prof = (r[harga] / median_bulan).groupby(r[KOL_PASAR]).mean()
    prof.columns = [f'rasio_{c[2:]}' for c in prof.columns]
    prof['inflasi_rata'] = r.groupby(KOL_PASAR)[kol_inf].mean()
    prof['volatilitas_inflasi'] = r.groupby(KOL_PASAR)[kol_inf].std()
    r['rentang_harga'] = tambah_rentang(r)
    if r['rentang_harga'].notna().any():
        prof['rentang_harga'] = r.groupby(KOL_PASAR)['rentang_harga'].mean()
    prof = prof.fillna(prof.median())
    X = StandardScaler().fit_transform(prof)
    skor = {}
    for k in range(k_min, min(k_max, len(prof) - 1) + 1):
        skor[k] = silhouette_score(X, KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X))
    k_terbaik = max(skor, key=skor.get)
    prof['klaster'] = KMeans(n_clusters=k_terbaik, n_init=10, random_state=42).fit_predict(X) + 1
    rasio_cols = [c for c in prof.columns if c.startswith('rasio_')]
    prof['komposit_harga'] = prof[rasio_cols].mean(axis=1)
    ringkas = prof.groupby('klaster').agg(jumlah_pasar=('klaster', 'size'), komposit_harga=('komposit_harga', 'mean'),
                                          inflasi_rata=('inflasi_rata', 'mean'), volatilitas_inflasi=('volatilitas_inflasi', 'mean'))
    vol_tengah = prof['volatilitas_inflasi'].median()
    nama = {}
    for i, baris in ringkas.iterrows():
        harga_txt = ('harga jauh di atas median' if baris['komposit_harga'] > 1.10 else 'harga sedikit di atas median' if baris['komposit_harga'] > 1.03
                     else 'harga di bawah median' if baris['komposit_harga'] < 0.97 else 'harga sekitar median')
        vol_txt = 'volatil' if baris['volatilitas_inflasi'] >= vol_tengah else 'stabil'
        nama[i] = f'Klaster {i}: {harga_txt}, {vol_txt}'
    prof['label_klaster'] = prof['klaster'].map(nama)
    ringkas['label_klaster'] = ringkas.index.map(nama)
    return prof, ringkas, pd.Series(skor, name='silhouette')

## Jalankan klastering
Jumlah klaster dipilih dari 3 sampai 6 berdasarkan silhouette tertinggi.

In [ ]:
df = pd.read_csv(data_path, parse_dates=[KOL_TANGGAL])
prof, ringkas, sil = klaster_pasar(df)
print('Silhouette per jumlah klaster:')
print(sil.round(3).to_string())
print(f'Jumlah klaster terpilih: {prof["klaster"].nunique()}')

In [ ]:
ringkas.round(2)

## Profil tiap klaster (nilai rata-rata terstandar)

In [ ]:
fitur_profil = [c for c in prof.columns if c not in ('klaster', 'label_klaster', 'komposit_harga')]
z = (prof[fitur_profil] - prof[fitur_profil].mean()) / prof[fitur_profil].std()
profil = z.groupby(prof['klaster']).mean()
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(profil.values, cmap='RdBu_r', vmin=-2, vmax=2, aspect='auto')
ax.set_xticks(range(len(fitur_profil)))
ax.set_xticklabels(fitur_profil, rotation=60, ha='right')
ax.set_yticks(range(len(profil)))
ax.set_yticklabels([f'Klaster {i}' for i in profil.index])
plt.colorbar(im, ax=ax)
ax.set_title('Profil klaster pasar')
plt.savefig(f'{output_dir}/grafik_profil_klaster.png', dpi=150, bbox_inches='tight')
plt.show()

## Sebaran klaster di peta

In [ ]:
lokasi = df.drop_duplicates(KOL_PASAR).set_index(KOL_PASAR)[['lat', 'lon', KOL_PROVINSI]]
peta = prof.join(lokasi)
plt.figure(figsize=(12, 5))
sc = plt.scatter(peta['lon'], peta['lat'], c=peta['klaster'], cmap='tab10', s=35, edgecolor='k', linewidth=0.3)
plt.colorbar(sc, label='Klaster')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Tipologi pasar berdasarkan klaster')
plt.savefig(f'{output_dir}/grafik_peta_klaster.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
pd.crosstab(peta[KOL_PROVINSI], peta['klaster'])

## Simpan hasil

In [ ]:
prof.to_csv(f'{output_dir}/klaster_pasar.csv')
ringkas.to_csv(f'{output_dir}/ringkasan_klaster.csv')
print('Hasil klastering tersimpan di', output_dir)